# RAG (Retrival Augmented Generation)

In [ ]:
# import library

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.readers.file import PyMuPDFReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from dotenv import load_dotenv
from llama_index.core.response.pprint_utils import pprint_response, pprint_metadata, pprint_source_node
from llama_index.core.settings import Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine
load_dotenv()


True

In [ ]:
# set embedding model and LLM

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm = OpenRouter(
    api_key="Your_api_key",
    model="openai/gpt-oss-20b"
)

2026-10-02 18:49:11,595 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-02 18:49:11,749 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-02 18:49:12,077 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-02 18:49:12,168 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentenc

In [28]:
# read pdf file

reader = PyMuPDFReader()
document = reader.load_data(file_path="report.pdf")

In [29]:
len(document)

257

### apply Indexing 
#### it handle automatically, chunking + embedding + vector store

##### here we have options that which vector database we can use (chorma db, pinecone, faiss)

In [31]:
index = VectorStoreIndex.from_documents(documents=document, show_progress=True)

Generating embeddings: 100%|██████████| 257/257 [00:20<00:00, 12.36it/s]


#### Retrival/Query engine

In [32]:
query_engine = index.as_query_engine(similarity_top_k=6)

#### LLM Response

In [33]:
res = query_engine.query('Advantage of White Box Testing')
print(res.response) #llm response text
print(res.metadata) #metadata

2026-10-02 18:59:32,682 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


**Advantages of White‑Box Testing**

- **Complete path coverage** – ensures every independent execution path is examined at least once.  
- **Decision‑level validation** – checks all logical branches for both true and false outcomes.  
- **Loop boundary testing** – executes loops at their minimum, maximum, and boundary values to uncover off‑by‑one or infinite‑loop issues.  
- **Internal structure inspection** – verifies that data structures, control flow, and internal logic behave as intended.  
- **Early error detection** – can reveal logical, typographical, or boundary‑condition errors that are unlikely to surface during black‑box testing.  
- **Systematic traceability** – provides a clear mapping between code and test cases, facilitating maintenance and regression testing.
{'2c3f4579-2aa6-4e83-9b46-8efde030ed54': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '24'}, '4649e24b-a541-4fa9-a01e-017253b8d3c7': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '234'}, '

#### increase retrive accuracy
#### postprocessors

In [41]:
retriver = VectorIndexRetriever(index=index, similarity_top_k=4, verbose=True)

#appy threshold
processor = SimilarityPostprocessor(similarity_cutoff=0.40)

#### retrive

In [42]:
query_engine = RetrieverQueryEngine(retriever=retriver, node_postprocessors=[processor])

In [ ]:
res = query_engine.query('what is Unit Testing')

2026-10-02 19:05:23,884 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Unit testing is a technique that examines each individual module or component in isolation. It verifies that the unit behaves correctly by checking its control paths, input‑output interfaces, data flow, boundary conditions, and ensuring that all independent execution paths are exercised.
{'c543ddb4-a634-48a7-ad29-eea7669b4674': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '233'}, '8a1152c7-1144-4028-8a9f-651680747112': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '23'}, 'e9630eca-3e93-49d9-9e8b-880a42ac86dc': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '239'}, '4649e24b-a541-4fa9-a01e-017253b8d3c7': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '234'}}


In [44]:
print(res.response)
print(res.metadata)

Unit testing is a technique that examines each individual module or component in isolation. It verifies that the unit behaves correctly by checking its control paths, input‑output interfaces, data flow, boundary conditions, and ensuring that all independent execution paths are exercised.
{'c543ddb4-a634-48a7-ad29-eea7669b4674': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '233'}, '8a1152c7-1144-4028-8a9f-651680747112': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '23'}, 'e9630eca-3e93-49d9-9e8b-880a42ac86dc': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '239'}, '4649e24b-a541-4fa9-a01e-017253b8d3c7': {'total_pages': 257, 'file_path': 'report.pdf', 'source': '234'}}


### complete